### Import required Python packages

In [1]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from math import nan
from pathlib import Path
import dask
import yaml
import os
import warnings
import gc
import textwrap
warnings.filterwarnings('ignore')
import glob

### Configure and launch a Dask cluster on Casper

This cell:

- Creates a Dask `PBSCluster` on **Casper** with the requested project code, walltime, and network interface.
- Scales the cluster to **24 workers** so computations can run in parallel.
- Sets the Dask **dashboard URL** so it can be viewed from JupyterHub.
- Creates a Dask `Client` connected to this cluster, so later cells can run computations in parallel using Dask.



In [7]:
cluster

PBSCluster(b2cdc6ea, 'tcp://10.18.206.68:36801', workers=24, threads=24, memory=670.56 GiB)

In [2]:
from dask_jobqueue import PBSCluster
from dask.distributed import Client
dask.config.set({"distributed.scheduler.worker_saturation":1.0})

cluster = PBSCluster(
    cores=1,
    memory='30GB',
    processes=1,
    queue='casper',
    local_directory='$TMPDIR',
    resource_spec='select=1:ncpus=1:mem=30GB',
    project='CESM0024',
    walltime='24:00:00',
    interface='mgt',
    job_extra_directives=[
        "-o dask_logs/dask-worker.$PBS_JOBID.out",
        "-e dask_logs/dask-worker.$PBS_JOBID.err",
    ]
)

# scale up
cluster.scale(24)

# change your urls to the dask dashboard so that you can see it
dask.config.set({'distributed.dashboard.link':'https://jupyterhub.hpc.ucar.edu/stable/user/{USER}/proxy/{port}/status'})

# Setup your client
client = Client(cluster)

In [5]:
cluster

Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/hannay/proxy/35485/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://10.18.206.68:36801,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/hannay/proxy/35485/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [17]:
cluster.close()
client.close()

### summary of the cluster state: configuration, workers, dashboard link, etc.

In [6]:
cluster

Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/hannay/proxy/41415/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://10.18.206.69:39233,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/hannay/proxy/41415/status,Total threads: 0
Started: Just now,Total memory: 0 B


### Collect input file lists

This cell uses `glob` to search for all the CAM history files needed for the processing workflow:

- `h5a`: all files matching `*.h5a.*`  
- `h5i`: all files matching `*.h5i.*`

Both lists are sorted to ensure the files are processed in the correct chronological order.  
These lists will be used later to loop over matching `h5a` / `h5i` file pairs.

In [10]:

h5a = sorted(glob.glob("/glade/campaign/cesm/community/amwg/cam_hr/f.e30_cam6_4_116.FHISTC_LTso.ne120_L58.c6aero.001/atm/hist/*.h5a.*2000*"))
h5i = sorted(glob.glob("/glade/campaign/cesm/community/amwg/cam_hr/f.e30_cam6_4_116.FHISTC_LTso.ne120_L58.c6aero.001/atm/hist/*.h5i.*2000*"))


### Convert PRECT, compute Tb, and create merged hourly CAM files

This cell loops over all matching `h5a` / `h5i` CAM history files and creates new merged hourly NetCDF files with consistent variables for later analysis.

What it does step by step:

- Defines constants and options:
  - `sigma`, `a`, `b` for computing the flux-equivalent brightness temperature **Tb** from **FLUT**
  - `h5avars = ['PRECT']` to keep only PRECT from the `h5a` files
  - Sets the output directory `outdir` and creates it if needed

- Loops over all `h5a` / `h5i` file pairs:
  - Extracts a date string from the filename and builds an output name like  
    `MERGED_CAM7_L58_HR_hourly.YYYY-MM-DD.nc`
  - Skips the file if the merged output already exists (avoids re-doing work)

- For each pair of files:
  - **From `h5a`**:
    - Opens the dataset with Dask chunking
    - Keeps only `PRECT`
    - Converts `PRECT` units from m/s to **mm/hr**
    - Preserves metadata and updates the units attribute

  - **From `h5i`**:
    - Opens the dataset with Dask chunking
    - Reads `FLUT` and computes **Tb** using the radiative transfer approximation  
      \[
      Tf = (FLUT / \sigma)^{1/4}, \quad
      Tb = \frac{-a + \sqrt{a^2 + 4 b Tf}}{2 b}
      \]
    - Adds `Tb` as a new variable with appropriate name and metadata

  - Ensures the time axis in `h5adat` matches the `h5i` time
  - Merges `h5a` and `h5i` content into a single `xarray.Dataset`
  - Writes the merged dataset to NetCDF (`NETCDF4` format) in `outdir`
  - Closes all open datasets to free memory

Overall, this cell prepares a clean set of **merged hourly CAM files** containing converted `PRECT`, original `h5i` fields, and the newly computed `Tb`, ready for downstream diagnostics.

In [11]:
%%time
# wmodified to add Tb, 2D X/Y, and proper PRECT/lat/lon metadata

import os
import numpy as np
import xarray as xr

# Constants for Tb calculation
sigma = 5.67e-8        # W m-2 K-4
a = 1.228
b = -1.106e-3          # K-1

h5avars = ['PRECT']

outdir = "/glade/derecho/scratch/hannay/MOAPP/hourly_files/"
os.makedirs(outdir, exist_ok=True)

for ifile in np.arange(0, len(h5a), 1):
# for ifile in np.arange(0, 1, 1):  # debug with a single file
    print("=====")

    fname = os.path.basename(h5i[ifile])

    # Extract timestamp: '2000-01-11' from '2000-01-11-03600'
    timestamp = fname.split(".")[-2]        # '2000-01-11-03600'
    date_str = timestamp.rsplit("-", 1)[0]  # '2000-01-11'

    # Build your new filename
    new_fname = f"CAM7_L58_HR_hourly.{date_str}.nc"
    outpath = os.path.join(outdir, new_fname)

    # 👉 Skip if it already exists
    if os.path.exists(outpath):
        print(f"File already exists, skipping: {outpath}")
        continue

    print(f"Creating: {outpath}")

    # --- Open h5a and convert PRECT ---
    h5adat = xr.open_dataset(h5a[ifile], chunks={"time": 24}, engine="netcdf4")
    h5adat = h5adat[h5avars]

    # m/s → mm/hr and cast to float32
    h5adat['PRECT'] = (h5adat['PRECT'] * 1000.0 * 3600.0).astype('float32')

    # Set PRECT attributes exactly as desired
    h5adat['PRECT'].attrs = {
        "long_name": "Total (convective and large-scale) precipitation rate (liq + ice)",
        "units": "mm hr-1",
        "cell_methods": "time: mean",
    }
    # _FillValue = NaNf
    h5adat['PRECT'].encoding['_FillValue'] = np.float32(np.nan)

    # --- Open h5i and compute Tb ---
    h5idat = xr.open_dataset(h5i[ifile], chunks={"time": 24}, engine="netcdf4")
    FLUT = h5idat['FLUT']

    Tf = (FLUT / sigma) ** 0.25
    Tb = (-a + np.sqrt(a**2 + 4.0 * b * Tf)) / (2.0 * b)

    # Give Tb a name & metadata
    Tb = Tb.rename('Tb')
    Tb.attrs['long_name'] = "flux equivalent brightness temperature"
    Tb.attrs['units'] = "K"

    # Add Tb into h5idat so it will be included in the merge
    h5idat['Tb'] = Tb

    # Use the time axis from h5i
    h5adat['time'] = h5idat.time.values

    # Merge the two datasets
    newdat = xr.merge([h5adat, h5idat])

    # --- Rename dimensions lat -> y, lon -> x (dims & coord dims) ---
    rename_dict = {}
    if 'lat' in newdat.dims:
        rename_dict['lat'] = 'y'
    if 'lon' in newdat.dims:
        rename_dict['lon'] = 'x'
    if rename_dict:
        newdat = newdat.rename(rename_dict)

    # --- Create 2D X(y, x) and Y(y, x) ---

    # Latitude coordinate (1D)
    if 'lat' in newdat:
        lat_1d = newdat['lat']
    elif 'y' in newdat:
        lat_1d = newdat['y']
    else:
        raise KeyError("No latitude coordinate found (expected 'lat' or 'y').")

    # Longitude coordinate (1D)
    if 'lon' in newdat:
        lon_1d = newdat['lon']
    elif 'x' in newdat:
        lon_1d = newdat['x']
    else:
        raise KeyError("No longitude coordinate found (expected 'lon' or 'x').")

    # Broadcast to 2D (y, x): Y(y,x) from lat, X(y,x) from lon
    Y_2d, X_2d = xr.broadcast(lat_1d, lon_1d)
    Y_2d = Y_2d.rename('Y')
    X_2d = X_2d.rename('X')

    # Metadata for X/Y
    Y_2d.attrs['long_name'] = 'latitude on (y,x) grid'
    Y_2d.attrs['units'] = lat_1d.attrs.get('units', 'degrees_north')

    X_2d.attrs['long_name'] = 'longitude on (y,x) grid'
    X_2d.attrs['units'] = lon_1d.attrs.get('units', 'degrees_east')

    # Attach to dataset
    newdat['Y'] = Y_2d
    newdat['X'] = X_2d

    # --- Create 1D lat(y) and lon(x) variables with desired attrs ---

    # Values for y/x
    if 'y' in newdat.coords:
        y_vals = newdat['y'].values
    else:
        y_vals = lat_1d.values

    if 'x' in newdat.coords:
        x_vals = newdat['x'].values
    else:
        x_vals = lon_1d.values

    # Drop y/x coordinate variables (keep dims 'y' and 'x')
    drop_vars = [v for v in ['y', 'x'] if v in newdat.coords]
    if drop_vars:
        newdat = newdat.drop_vars(drop_vars)

    # Define lat(y) and lon(x)
    lat_da = xr.DataArray(
        y_vals,
        dims=('y',),
        name='lat',
        attrs={
            'long_name': 'latitude',
            'units': 'degrees_north',
        },
    )
    lon_da = xr.DataArray(
        x_vals,
        dims=('x',),
        name='lon',
        attrs={
            'long_name': 'longitude',
            'units': 'degrees_east',
        },
    )

    # Set FillValue for lat/lon (NaN as double)
    lat_da.encoding['_FillValue'] = np.nan
    lon_da.encoding['_FillValue'] = np.nan

    # Attach them as coordinates
    newdat = newdat.assign_coords(lat=lat_da, lon=lon_da)

    # --- Write out ---
    newdat.to_netcdf(outpath, format='NETCDF4')
    newdat.close()
    h5adat.close()
    h5idat.close()


=====
File already exists, skipping: /glade/derecho/scratch/hannay/MOAPP/hourly_files/CAM7_L58_HR_hourly.2000-01-11.nc
=====
File already exists, skipping: /glade/derecho/scratch/hannay/MOAPP/hourly_files/CAM7_L58_HR_hourly.2000-01-26.nc
=====
Creating: /glade/derecho/scratch/hannay/MOAPP/hourly_files/CAM7_L58_HR_hourly.2000-02-10.nc
=====
Creating: /glade/derecho/scratch/hannay/MOAPP/hourly_files/CAM7_L58_HR_hourly.2000-02-25.nc
=====
Creating: /glade/derecho/scratch/hannay/MOAPP/hourly_files/CAM7_L58_HR_hourly.2000-03-12.nc
=====
Creating: /glade/derecho/scratch/hannay/MOAPP/hourly_files/CAM7_L58_HR_hourly.2000-03-27.nc
=====
Creating: /glade/derecho/scratch/hannay/MOAPP/hourly_files/CAM7_L58_HR_hourly.2000-04-11.nc
=====
Creating: /glade/derecho/scratch/hannay/MOAPP/hourly_files/CAM7_L58_HR_hourly.2000-04-26.nc
=====
Creating: /glade/derecho/scratch/hannay/MOAPP/hourly_files/CAM7_L58_HR_hourly.2000-05-11.nc
=====
Creating: /glade/derecho/scratch/hannay/MOAPP/hourly_files/CAM7_L58_HR

# ====== STOP ==== 

In [ ]:
cluster.close()
client.close()
